# CS 521 HW3 Problem 1
NetID: ws39

I train a standard model and an L-infinity adversarial model for five epochs from the same initialization. The model and training settings come from the [Problem 3 starter](https://github.com/nalinwadhwa02/CCS521FA26HW/blob/main/hw3/q3/multi_norm_adversarial_training_cifar10.ipynb). PGD is adapted from my HW1 implementation with a random start.

To run in Colab, select a GPU and run the cells in order. The first cell downloads the data. Both models are trained from scratch.


In [ ]:
%pip -q install gdown
import gdown
from pathlib import Path

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
for file_id, name in [
    ('1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe', 'cifar10_train_10k.pt'),
    ('1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA', 'cifar10_test_1k.pt'),
]:
    path = data_dir / name
    if not path.exists():
        gdown.download(id=file_id, output=str(path), quiet=False)


## Model and data helpers


In [ ]:
"""CS 521 HW3 Problem 1. Adapted from the supplied Problem 3 notebook."""
import json
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]


## PGD, training, and evaluation
Training uses 10-step PGD at epsilon 8/255. For evaluation, I use 50 steps and 2 restarts, with step size 2.5 * epsilon / steps. FGSM takes one step from the clean image. An image counts as robust only if the clean prediction and all attacked predictions are correct.


In [ ]:
def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = (x + delta).clamp(0, 1) - x
    for _ in range(steps):
        delta = delta.detach().requires_grad_(True)
        loss = F.cross_entropy(model(x + delta), y)
        grad, = torch.autograd.grad(loss, delta)
        delta = (delta.detach() + alpha * grad.sign()).clamp(-eps, eps)
        delta = (x + delta).clamp(0, 1) - x
    return (x + delta).detach()


def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def single_norm_loss(model, x, y, eps=8/255):
    model.eval()
    x_adv = pgd_linf(model, x, y, eps, 2.5 * eps / 10, 10)
    model.train()
    return F.cross_entropy(model(x_adv), y)


def synchronize():
    if device.type == 'mps':
        torch.mps.synchronize()
    elif device.type == 'cuda':
        torch.cuda.synchronize()


def adv_train(model, loss_fn, x_train, y_train, output, epochs=5, lr=0.01, batch_size=128, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    history = []
    for epoch in range(epochs):
        synchronize()
        start = time.perf_counter()
        losses = []
        for batch_index, (x, y) in enumerate(batches(x_train, y_train, batch_size, shuffle=True)):
            if len(y) < batch_size:
                continue
            x = augment(x)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model, x, y)
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
            if (batch_index + 1) % 10 == 0:
                print(f'epoch {epoch+1}, batch {batch_index+1}/78', flush=True)
        synchronize()
        row = {'epoch': epoch+1, 'loss': float(np.mean(losses)), 'seconds': time.perf_counter()-start}
        history.append(row)
        print(row, flush=True)
        torch.save(model.state_dict(), output / 'model.pt')
        (output / 'training.json').write_text(json.dumps(history, indent=2))
    return model.eval(), history


def evaluate(model, x_test, y_test, eps, steps=50, restarts=2, batch_size=125):
    model.eval()
    torch.manual_seed(0)
    counts = {'clean': 0, 'fgsm': 0, 'pgd': 0}
    for x, y in batches(x_test, y_test, batch_size, shuffle=False):
        with torch.no_grad():
            clean = model(x).argmax(1).eq(y)
        if eps == 0:
            fgsm = robust = clean
        else:
            x_fgsm = pgd_linf(model, x, y, eps, eps, 1, rand_init=False)
            with torch.no_grad():
                fgsm = clean & model(x_fgsm).argmax(1).eq(y)
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = pgd_linf(model, x, y, eps, 2.5*eps/steps, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1).eq(y)
        for key, mask in [('clean', clean), ('fgsm', fgsm), ('pgd', robust)]:
            counts[key] += mask.sum().item()
    return {'epsilon': eps, **{k: 100*v/len(y_test) for k,v in counts.items()}, 'counts': counts}


def load_data(path):
    x, y = torch.load(path, map_location='cpu', weights_only=True)
    return x.float().div(255).to(device), y.to(device)



In [ ]:
torch.set_num_threads(4)
x_train, y_train = load_data(data_dir / 'cifar10_train_10k.pt')
x_test, y_test = load_data(data_dir / 'cifar10_test_1k.pt')
assert x_train.shape == (10000, 3, 32, 32)
assert x_test.shape == (1000, 3, 32, 32)
assert torch.bincount(y_train).tolist() == [1000] * 10
print(device, x_train.shape, x_test.shape)


## (a) Train both models
Both models use seed 0. I use the starter settings: SGD, learning rate 0.01, momentum 0.9, weight decay 0.0005, batch size 128, and cosine decay. The last incomplete batch is skipped.


In [ ]:
models = {}
histories = {}
for name, loss_fn in [('standard', standard_loss), ('linf', single_norm_loss)]:
    output = Path('results') / name
    output.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(0)
    model = make_model()
    models[name], histories[name] = adv_train(model, loss_fn, x_train, y_train, output)


## (b) Compare FGSM and PGD
I attack each model using its own gradients. The “original model” is the standard model from part (a).


In [ ]:
results = {}
for name, model in models.items():
    results[name] = []
    for numerator in [0, 2, 4, 8, 12, 16]:
        row = evaluate(model, x_test, y_test, numerator / 255)
        results[name].append(row)
        print(name, numerator, row)
        (Path('results') / name / 'evaluation.json').write_text(json.dumps(results[name], indent=2))


In [ ]:
import pandas as pd
rows = []
for i, numerator in enumerate([0, 2, 4, 8, 12, 16]):
    a, b = results['standard'][i], results['linf'][i]
    rows.append({'epsilon': f'{numerator}/255', 'Standard FGSM (%)': a['fgsm'],
                 'Standard PGD (%)': a['pgd'], 'Adversarial FGSM (%)': b['fgsm'],
                 'Adversarial PGD (%)': b['pgd']})
display(pd.DataFrame(rows))


## Local results

I ran these experiments on Apple M3 (MPS) with PyTorch 2.8.0 and seed 0. Colab results may differ.

| epsilon | Standard FGSM | Standard PGD | Adversarial FGSM | Adversarial PGD |
|---|---:|---:|---:|---:|
| 0/255 | 57.9% | 57.9% | 38.5% | 38.5% |
| 2/255 | 26.6% | 21.6% | 34.4% | 34.3% |
| 4/255 | 11.6% | 6.2% | 29.9% | 30.0% |
| 8/255 | 3.2% | 0.5% | 24.5% | 23.4% |
| 12/255 | 1.3% | 0.0% | 19.4% | 17.7% |
| 16/255 | 0.9% | 0.0% | 15.8% | 12.9% |
